In [ ]:
#r "nuget: Azure.AI.Projects, 1.2.0-beta.5"
#r "nuget: Azure.Identity, 1.18.0-beta.2"
#r "nuget: Microsoft.Agents.AI, 1.0.0-rc1"
#r "nuget: Microsoft.Agents.AI.AzureAI, 1.0.0-rc1"

using Azure.AI.Projects;
using Azure.Identity;
using Microsoft.Agents.AI;
using System;
using System.Threading;
using System.Threading.Tasks;
using System.Collections.Generic;
using Microsoft.Extensions.AI;

// Agent with Local Function Tools to get time.

var endpointUrl = Environment.GetEnvironmentVariable("AZURE_FOUNDRY_PROJECT_ENDPOINT");
var endpoint = endpointUrl != null ? new Uri(endpointUrl) : throw new InvalidOperationException("AZURE_FOUNDRY_PROJECT_ENDPOINT environment variable is not set.");
var deploymentName = "gpt-4o";
var apiKey = Environment.GetEnvironmentVariable("AZURE_FOUNDRY_PROJECT_API_KEY") ?? throw new InvalidOperationException("AZURE_FOUNDRY_PROJECT_API_KEY environment variable is not set.");
var githubToken = Environment.GetEnvironmentVariable("GITHUBTOKEN") ?? throw new InvalidOperationException("GITHUB_TOKEN environment variable is not set.");

const string agentName = "AgentFrameworkAgent-1";

var credOptions = new DefaultAzureCredentialOptions
{
TenantId = "<TenantId>" // Specify the desired tenant ID
};  

// Get a client to create/retrieve/delete server side agents with Azure Foundry Agents.
AIProjectClient aiProjectClient = new AIProjectClient(new Uri(endpointUrl), new DefaultAzureCredential(credOptions));

AIAgent timeAgent = await aiProjectClient.CreateAIAgentAsync(name: agentName, model: deploymentName, instructions: "You are a time expert", tools
: new List<AITool>()
{
    AIFunctionFactory.Create(Tools.GetCurrentDateAndTime, "current_date_and_time"),
    AIFunctionFactory.Create(Tools.GetCurrentTimeZone, "current_timezone")
});

AgentSession session = await timeAgent.CreateSessionAsync(CancellationToken.None);

Console.Write("Time Tool > ");
ChatMessage message = new ChatMessage(ChatRole.User, "What is the current date and time, and timezone?");
var timeResponse = await timeAgent.RunAsync(message, session);
Console.WriteLine(timeResponse);


// Simple Tools helper providing the methods referenced below.
public static class Tools
{
    public static Task<string> GetCurrentDateAndTime() =>
        Task.FromResult(DateTime.UtcNow.ToString("o"));

    public static Task<string> GetCurrentTimeZone() =>
        Task.FromResult(TimeZoneInfo.Local.StandardName);
}

Installed Packages azure.ai.projects, 1.2.0-beta.5 Azure.Identity, 1.18.0-beta.2 Microsoft.Agents.AI, 1.0.0-rc1 Microsoft.Agents.AI.AzureAI, 1.0.0-rc1

Time Tool > The current date and time is March 3, 2026, 04:18:55 UTC. The timezone is Central Standard Time.



warning CS1702: Assuming assembly reference 'System.ClientModel, Version=1.8.0.0, Culture=neutral, PublicKeyToken=92742159e12e44c8' used by 'Azure.Core' matches identity 'System.ClientModel, Version=1.8.1.0, Culture=neutral, PublicKeyToken=92742159e12e44c8' of 'System.ClientModel', you may need to supply runtime policy

